# M2 · Harness de evaluación de tres dimensiones

Corre sobre las salidas que M1 dejó en `results/m1_predicciones.json`: no
regenera nada, así que es barato y reproducible. Va en CPU.

Las tres dimensiones y lo que **no** mide cada una:

| | mide | no mide |
|---|---|---|
| **1 · clásica** | similitud con el gold: ROUGE-L, embeddings, accuracy, macro-F1 | si el derecho está bien aplicado. Una respuesta elegante y equivocada puntúa alto |
| **2 · LLM-juez** | calidad del razonamiento, que ninguna métrica léxica ve | no es un oráculo: arrastra sesgos que hay que medir aparte |
| **3 · dominio** | acierto jurídico, alucinación normativa, pertinencia de las citas, coherencia | nada de esto se puede leer de un número de similitud |

En una frase: **revisa si el sistema eligió el mecanismo correcto, si un
segundo criterio independiente lo considera bien razonado, y si las normas que
cita existen y vienen al caso.**

La logica vive en `amparo/pipelines/m2_harness.py` (una funcion por etapa) y `amparo/evaluacion/acuerdo.py`. Este notebook solo las llama en orden.

In [ ]:
!pip install -q groq==1.7.0

## 0 · Arranque

Monta Drive y pone la raiz en `sys.path`: es lo unico que no puede vivir en el paquete, porque hasta aqui `amparo` no se puede importar.

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass          # fuera de Colab: el repo ya esta en disco

CANDIDATAS = [Path("/content/drive/MyDrive/agenteAmparo"),
              Path("/content/drive/MyDrive/Colab Notebooks/agenteAmparo"),
              Path("/content/agenteAmparo"), Path("."), Path("..")]
RAIZ = next((p.resolve() for p in CANDIDATAS
             if (p / "amparo" / "nucleo" / "schema.py").exists()), None)
if RAIZ is None:
    raise SystemExit("No encuentro el proyecto: ninguna de "
                     + ", ".join(map(str, CANDIDATAS))
                     + " tiene amparo/nucleo/schema.py. ¿Drive montado y la "
                     "carpeta agenteAmparo subida?")
sys.path.insert(0, str(RAIZ))
print("raiz:", RAIZ)

`m2.preparar()` corre `entorno.preparar("m2")`: corrida nueva, sello contra `data/subida_sellada.txt`, la clave de Groq (por teclado si falta: M2 no sirve sin juez) y `arranque.celda("m2")`, en ese orden.

In [ ]:
from amparo.pipelines import m2_harness as m2

ctx = m2.preparar()

## 1 · Las salidas de M1

Lee `results/m1_predicciones.json` (con la corrida de M1 que las produjo) y aplica el recorte defensivo de colas degeneradas.

In [ ]:
m2.cargar_salidas(ctx)

## 2 · Formato contra vocabulario

Imitar la **forma** de la respuesta y usar el **vocabulario cerrado** correcto son dos cosas distintas.

In [ ]:
m2.formato_vocabulario(ctx)

## 3 · Dimensión 1 — la métrica clásica y su límite

In [ ]:
m2.rouge_vs_acierto(ctx)

## 4 · Dimensión 2 — LLM-as-a-judge

Cinco decisiones de diseño del juez, y cada una responde a un sesgo conocido:

1. **Otra familia de modelos.** El sistema evaluado es Qwen2.5-7B; el juez es
   `openai/gpt-oss-120b`. Los modelos Qwen de la cuenta quedan excluidos a
   propósito: un juez de la misma familia reintroduce la auto-preferencia.
2. **Se valida antes de cachear.** Un JSON truncado que se cachea queda
   congelado y se lee como "empate" para siempre.
3. **Un fallo de lectura devuelve `None`**, no un resultado plausible.
4. **La rúbrica lleva defensa contra inyección de prompt**, porque el relato lo
   escribe un tercero y entra al juez dentro del dato.
5. **El material evaluado va delimitado**, entre marcas explícitas y con una
   línea que cierra el dato. La defensa del punto 4 le **dice** al juez que lo
   de abajo es dato; esto se lo **muestra**. Hacía falta: sin una frontera, el
   modelo tiene que inferir dónde acaba la instrucción, y las dos inyecciones
   que funcionaron en `m2/20261019T034128Z` eran justamente «falsa política de
   evaluación» —tono de sistema aprovechando esa ambigüedad—. El envoltorio
   está en **una** función (`juez._envolver`) usada por los tres sitios que
   arman el mensaje: tres copias divergirían, y además la clave de caché
   incluye el mensaje, así que partirían la caché en tres.

**La rúbrica se escribe una vez y se usa en dos sitios.** `data/validacion_juez.py`
extrae la escala de `juez.RUBRICA` en vez de copiarla al formulario del
abogado. Si el humano puntuara con otra definición de «3», el desacuerdo
medido sería el de las dos escalas y no el de los dos criterios —y la cifra no
diría nada sobre el juez.

In [ ]:
m2.elegir_juez(ctx)

### Sesgo 1 · posición

In [ ]:
m2.sesgo_posicion(ctx)

### Sesgo 2 · longitud

In [ ]:
m2.sesgo_longitud(ctx)

### Sesgo 3 · auto-preferencia — **medido**

Condicionado en la verdad de dominio: en `empate_correctas` las dos respuestas aciertan, así que cualquier preferencia sistemática **es sesgo**. Ver el docstring de `m2.autopreferencia`.

In [ ]:
m2.autopreferencia(ctx)

### Sesgo 4 · inyección de prompt

Cada sonda pide una nota **distinta** de la que merece la respuesta, en las dos direcciones. Se mide en los dos órdenes y no se promedia.

In [ ]:
m2.inyeccion(ctx)

### Sonda 5 · ¿el juez distingue una respuesta correcta de una rota? · **validez**

Validez de construcción, no acuerdo con un humano: M2-1b sigue abierto.

In [ ]:
m2.validez(ctx)

### ¿La nota es del sistema o del juez? · acuerdo entre dos jueces

La familia del segundo juez importa más que su tamaño. Si ningún segundo juez responde, **no se publica nada**. Ver `amparo/evaluacion/acuerdo.py`.

In [ ]:
m2.acuerdo_entre_jueces(ctx)

## 5 · El harness completo

In [ ]:
m2.harness_completo(ctx)

## 6 · Reproducibilidad

Dentro de la sesión (el harness dos veces da lo mismo) y entre corridas distintas guardadas en disco.

In [ ]:
m2.reproducibilidad(ctx)

## 7 · Scorecard

Comprueba **todos** los artefactos antes de publicar **ninguno**.

In [ ]:
m2.publicar(ctx)

import pandas as pd
pd.read_csv("results/scorecard_baseline.csv").set_index("sistema").T

## 8 · Qué debilidad revela el scorecard

Rellenar con los números de la corrida. Lo que hay que mirar sin maquillar:

1. **`acc_mecanismo` contra la clase mayoritaria.** Si el delta es pequeño, el
   fine-tuning no aportó en el campo que decide la respuesta.
2. **Gold contra adversarial.** La brecha es la medida de si el sistema razona
   o reconoce formas.
3. **Las dos cifras de alucinación.** La del campo `fundamento` y la de toda la
   salida. La diferencia son normas que el modelo afirma en la línea de
   procedencia, donde antes no mirábamos.
4. **`precision_citas` junto a `tasa_alucinacion`.** Una cita puede existir y no
   venir al caso; la primera métrica lo ve y la segunda no.
5. **El kappa del juez contra el dominio, leído junto a la adherencia.** Un
   kappa bajo no dice que el juez esté roto, y **subirlo no sería una
   mejora**: un juez que coincidiera con `acierto_juridico` sería una copia
   ruidosa de esa métrica y no aportaría el razonamiento, que es exactamente
   lo que la accuracy no ve. Si sale `indefinido`, es que los dos anotadores
   usaron una sola categoría y el estadístico no aplica.

   La cifra que sí tiene respuesta correcta es la de al lado:
   `adherencia_a_la_rubrica`. La rúbrica define `nota >= 3` como «mecanismo
   correcto», y eso se comprueba contra el gold sin preguntarle a nadie. Un 4
   sobre una respuesta con el mecanismo equivocado no es generosidad: es
   incumplir la escala que se le dio. Las dos direcciones se cuentan por
   separado porque cuestan distinto —**generosa** infla la dimensión 2,
   **severa** solo castiga una respuesta correcta—. Leídas juntas: kappa baja
   con adherencia **alta** significa que el juez mide algo que la accuracy no
   ve, que es para lo que existe; kappa baja con adherencia **baja**
   significaría que sus notas no quieren decir lo que la rúbrica dice, y
   entonces la dimensión 2 entera no se puede leer con esa escala.

6. **Las cinco sondas, y la quinta no mide lo mismo que las otras.** Posición,
   longitud, auto-preferencia e inyección miden si al juez se le puede
   **engañar**. Ninguna comprueba lo primero: que el instrumento **mida
   algo**. Un juez que puntuara al azar las pasaría todas —sin sesgo de
   posición, sin premiar la longitud, sin preferir a su linaje, sin obedecer
   inyecciones— y sus notas no valdrían nada. La sonda de validez le pone
   delante el gold y el mismo gold con un defecto aislado y mira si la nota
   baja.

   La auto-preferencia ya **no** está «evitada por construcción»: está
   **medida**. Se clasifican todos los pares gratis y el presupuesto de
   llamadas se gasta en orden de prioridad, empezando por los empates, que son
   los informativos. El artefacto publica `disponibles_por_estrato` y
   `seleccion`, así que la cifra se lee sabiendo de qué pares salió.

7. **Lo que sigue abierto: M2-1b.** Las cinco sondas y la adherencia son
   validez **de construcción**: dicen que el instrumento responde al defecto y
   aplica su escala. Ninguna dice que coincida con un abogado.
   `data/validacion_juez.py` escribe el formulario —misma rúbrica, con la nota
   del juez **oculta** para no anclar la del humano— y `--leer` calcula el
   acuerdo con `harness.acuerdo_entre_jueces`, la misma función que mide el
   acuerdo entre modelos. Es el único par del proyecto del que se puede
   afirmar que no comparte linaje con el juez.

   Y no se confunde con `data/revision_abogado_v3.json`, que validó el
   **gold**: un eval set impecable puntuado por un juez ciego da cifras
   inservibles igual.